# IR System — TF-IDF Retrieval
**Step 4:** Build TF-IDF index and run retrieval for both datasets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/ir_system_data'
import os, sys

if not os.path.exists('/content/ir-system'):
    !git clone https://github.com/ghazal-mohammad/ir-system.git /content/ir-system
else:
    !cd /content/ir-system && git pull
sys.path.insert(0, '/content/ir-system')
print('ready')

In [ ]:
import json, pickle
from services.indexing_service import load_index
from services.tfidf_service import build_tfidf_index, retrieve_tfidf, save_tfidf_data, load_tfidf_data
from services.preprocessing_service import preprocess

# Load inverted indexes
print('Loading indexes...')
index1 = load_index(f'{SAVE_DIR}/ct2021_index.pkl')
index2 = load_index(f'{SAVE_DIR}/msmarco_index.pkl')

with open(f'{SAVE_DIR}/ct2021_doc_lengths.json') as f:
    doc_lengths1 = json.load(f)
with open(f'{SAVE_DIR}/msmarco_doc_lengths.json') as f:
    doc_lengths2 = json.load(f)

print(f'CT2021: {len(index1):,} terms, {len(doc_lengths1):,} docs')
print(f'MSMARCO: {len(index2):,} terms, {len(doc_lengths2):,} docs')

In [ ]:
# Build TF-IDF index for CT2021
print('Building TF-IDF index for CT2021...')
tfidf1, idf1 = build_tfidf_index(index1, doc_lengths1)
print(f'CT2021 TF-IDF: {len(tfidf1):,} terms')
save_tfidf_data(tfidf1, idf1, f'{SAVE_DIR}/ct2021')
print('CT2021 done')

In [ ]:
# Build TF-IDF index for MSMARCO
print('Building TF-IDF index for MSMARCO...')
tfidf2, idf2 = build_tfidf_index(index2, doc_lengths2)
print(f'MSMARCO TF-IDF: {len(tfidf2):,} terms')
save_tfidf_data(tfidf2, idf2, f'{SAVE_DIR}/msmarco')
print('MSMARCO done')

In [ ]:
# Load queries for both datasets
import ir_datasets

ds1 = ir_datasets.load('clinicaltrials/2021/trec-ct-2021')
queries1 = {q.query_id: q.text for q in ds1.queries_iter()}
print(f'CT2021 queries: {len(queries1)}')

ds2 = ir_datasets.load('msmarco-passage/trec-dl-2019')
queries2 = {q.query_id: q.text for q in ds2.queries_iter()}
print(f'MSMARCO queries: {len(queries2)}')

In [ ]:
# Test TF-IDF retrieval on sample queries
sample_qid1 = list(queries1.keys())[0]
sample_query1 = queries1[sample_qid1]
tokens1 = preprocess(sample_query1)

results1 = retrieve_tfidf(tokens1, tfidf1, idf1, doc_lengths1, top_k=10)
print(f'CT2021 query: "{sample_query1}"')
print(f'Top 10 results:')
for r in results1:
    print(f'  rank {r["rank"]}: {r["doc_id"]} (score={r["score"]})')

print()
sample_qid2 = list(queries2.keys())[0]
sample_query2 = queries2[sample_qid2]
tokens2 = preprocess(sample_query2)

results2 = retrieve_tfidf(tokens2, tfidf2, idf2, doc_lengths2, top_k=10)
print(f'MSMARCO query: "{sample_query2}"')
print(f'Top 10 results:')
for r in results2:
    print(f'  rank {r["rank"]}: {r["doc_id"]} (score={r["score"]})')

In [ ]:
# Run full retrieval for all queries and save results
import json

print('Running TF-IDF retrieval on all CT2021 queries...')
all_results1 = {}
for qid, qtext in queries1.items():
    tokens = preprocess(qtext)
    all_results1[qid] = retrieve_tfidf(tokens, tfidf1, idf1, doc_lengths1, top_k=1000)
print(f'Done. {len(all_results1)} queries processed')

with open(f'{SAVE_DIR}/ct2021_tfidf_results.json', 'w') as f:
    json.dump(all_results1, f)
print('CT2021 results saved')

print('Running TF-IDF retrieval on all MSMARCO queries...')
all_results2 = {}
for qid, qtext in queries2.items():
    tokens = preprocess(qtext)
    all_results2[qid] = retrieve_tfidf(tokens, tfidf2, idf2, doc_lengths2, top_k=1000)
print(f'Done. {len(all_results2)} queries processed')

with open(f'{SAVE_DIR}/msmarco_tfidf_results.json', 'w') as f:
    json.dump(all_results2, f)
print('MSMARCO results saved')

print('\n=== TF-IDF Retrieval Complete ===')
print('Next: 05_retrieval_bm25.ipynb')